# Notebook 02 - Prétraitement des données

**Objectif** : Nettoyer, transformer et préparer le dataset Home Credit 
pour la modélisation. Ce notebook produit les jeux train/test prêts à 
être ingérés par les trois modèles (régression logistique, Random Forest, XGBoost).

## 0. Imports et chargement des données

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 50)
np.random.seed(42)

df = pd.read_csv("../data/raw/application_train.csv")
print(f"{df.shape[0]:,} lignes × {df.shape[1]} colonnes")

307,511 lignes × 122 colonnes


## 1. Suppression des colonnes trop incomplètes

Retrait des colonnes avec plus de 60 % de valeurs manquantes (17 colonnes identifiées au Notebook 01, essentiellement des caractéristiques immobilières).

In [7]:
# Identifier les colonnes avec >60% de manquants
seuil = 0.60
manquants_pct = df.isnull().mean()
colonnes_a_supprimer = manquants_pct[manquants_pct > seuil].index.tolist()

print(f"Colonnes supprimées ({len(colonnes_a_supprimer)}) :\n")
for col in colonnes_a_supprimer:
    print(f"  {col:35s} → {manquants_pct[col]:.1%} manquants")

# Suppression
df = df.drop(columns=colonnes_a_supprimer)
print(f"\nDataset après suppression : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

Colonnes supprimées (17) :

  OWN_CAR_AGE                         → 66.0% manquants
  YEARS_BUILD_AVG                     → 66.5% manquants
  COMMONAREA_AVG                      → 69.9% manquants
  FLOORSMIN_AVG                       → 67.8% manquants
  LIVINGAPARTMENTS_AVG                → 68.4% manquants
  NONLIVINGAPARTMENTS_AVG             → 69.4% manquants
  YEARS_BUILD_MODE                    → 66.5% manquants
  COMMONAREA_MODE                     → 69.9% manquants
  FLOORSMIN_MODE                      → 67.8% manquants
  LIVINGAPARTMENTS_MODE               → 68.4% manquants
  NONLIVINGAPARTMENTS_MODE            → 69.4% manquants
  YEARS_BUILD_MEDI                    → 66.5% manquants
  COMMONAREA_MEDI                     → 69.9% manquants
  FLOORSMIN_MEDI                      → 67.8% manquants
  LIVINGAPARTMENTS_MEDI               → 68.4% manquants
  NONLIVINGAPARTMENTS_MEDI            → 69.4% manquants
  FONDKAPREMONT_MODE                  → 68.4% manquants

Dataset après suppr

## 2. Traitement des valeurs aberrantes

Deux anomalies identifiées au Notebook 01 :
- DAYS_EMPLOYED : 55 374 occurrences de la valeur 365243 (code pour "sans emploi")
- CODE_GENDER : 4 occurrences de la modalité "XNA" (code pour "inconnu")

In [8]:
# DAYS_EMPLOYED : remplacer 365243 par NaN
n_aberrants = (df["DAYS_EMPLOYED"] == 365243).sum()
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
print(f"DAYS_EMPLOYED : {n_aberrants:,} valeurs aberrantes remplacées par NaN")

# CODE_GENDER : supprimer les lignes XNA
n_xna = (df["CODE_GENDER"] == "XNA").sum()
df = df[df["CODE_GENDER"] != "XNA"]
print(f"CODE_GENDER : {n_xna} lignes XNA supprimées")

print(f"\nDataset après nettoyage : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

DAYS_EMPLOYED : 55,374 valeurs aberrantes remplacées par NaN
CODE_GENDER : 4 lignes XNA supprimées

Dataset après nettoyage : 307,507 lignes × 105 colonnes


## 3. Imputation des valeurs manquantes restantes

- Variables numériques : imputation par la médiane
- Variables catégorielles : imputation par le mode

In [9]:
# Séparer numériques et catégorielles
colonnes_num = df.select_dtypes(include='number').columns.tolist()
colonnes_cat = df.select_dtypes(exclude='number').columns.tolist()

# Imputation des numériques par la médiane
n_manquants_num = df[colonnes_num].isnull().sum().sum()
for col in colonnes_num:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Imputation des catégorielles par le mode
n_manquants_cat = df[colonnes_cat].isnull().sum().sum()
for col in colonnes_cat:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"Valeurs manquantes imputées (numériques) : {n_manquants_num:,}")
print(f"Valeurs manquantes imputées (catégorielles) : {n_manquants_cat:,}")
print(f"Valeurs manquantes restantes : {df.isnull().sum().sum()}")

Valeurs manquantes imputées (numériques) : 5,085,379
Valeurs manquantes imputées (catégorielles) : 554,071
Valeurs manquantes restantes : 0


## 4. Création de features dérivées

Quatre variables à créer à partir des variables existantes :
- AGE : âge du client en années (dérivé de DAYS_BIRTH)
- CREDIT_INCOME_RATIO : ratio montant du crédit / revenu
- ANNUITY_INCOME_RATIO : ratio annuité / revenu
- EMPLOYMENT_YEARS : ancienneté professionnelle en années (dérivé de DAYS_EMPLOYED)

In [12]:
df['AGE'] = -df['DAYS_BIRTH'] / 365
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['EMPLOYMENT_YEARS'] = -df['DAYS_EMPLOYED'] / 365

print("Features dérivées créées :")
print(f"  AGE                  → min: {df['AGE'].min():.1f}, max: {df['AGE'].max():.1f}, médiane: {df['AGE'].median():.1f}")
print(f"  CREDIT_INCOME_RATIO  → min: {df['CREDIT_INCOME_RATIO'].min():.2f}, max: {df['CREDIT_INCOME_RATIO'].max():.2f}, médiane: {df['CREDIT_INCOME_RATIO'].median():.2f}")
print(f"  ANNUITY_INCOME_RATIO → min: {df['ANNUITY_INCOME_RATIO'].min():.4f}, max: {df['ANNUITY_INCOME_RATIO'].max():.4f}, médiane: {df['ANNUITY_INCOME_RATIO'].median():.4f}")
print(f"  EMPLOYMENT_YEARS     → min: {df['EMPLOYMENT_YEARS'].min():.1f}, max: {df['EMPLOYMENT_YEARS'].max():.1f}, médiane: {df['EMPLOYMENT_YEARS'].median():.1f}")

print(f"\nDataset après ajout des features : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

Features dérivées créées :
  AGE                  → min: 20.5, max: 69.1, médiane: 43.2
  CREDIT_INCOME_RATIO  → min: 0.00, max: 84.74, médiane: 3.27
  ANNUITY_INCOME_RATIO → min: 0.0002, max: 1.8760, médiane: 0.1628
  EMPLOYMENT_YEARS     → min: -0.0, max: 49.1, médiane: 4.5

Dataset après ajout des features : 307,507 lignes × 109 colonnes


## 5. Encodage des variables catégorielles

One-hot encoding avec suppression de la première modalité (drop_first=True) 
pour éviter la multicolinéarité, problématique pour la régression logistique.

In [13]:
colonnes_cat = df.select_dtypes(exclude='number').columns.tolist()
print(f"Colonnes catégorielles à encoder ({len(colonnes_cat)}) : {colonnes_cat}")

n_colonnes_avant = df.shape[1]
df = pd.get_dummies(df, columns=colonnes_cat, drop_first=True)

print(f"\nColonnes avant encodage : {n_colonnes_avant}")
print(f"Colonnes après encodage : {df.shape[1]}")
print(f"Colonnes créées : {df.shape[1] - n_colonnes_avant}")

Colonnes catégorielles à encoder (15) : ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']

Colonnes avant encodage : 109
Colonnes après encodage : 214
Colonnes créées : 105


## 6. Découpage train/test stratifié

Séparation 70/30, stratifiée sur TARGET pour préserver le ratio 92/8 
dans les deux sous-ensembles. random_state=42 pour la reproductibilité.

In [14]:
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

print(f"Jeu d'entraînement : {X_train.shape[0]:,} lignes × {X_train.shape[1]} colonnes")
print(f"Jeu de test         : {X_test.shape[0]:,} lignes × {X_test.shape[1]} colonnes")
print(f"\nTaux de défaut train : {y_train.mean():.2%}")
print(f"Taux de défaut test  : {y_test.mean():.2%}")

Jeu d'entraînement : 215,254 lignes × 212 colonnes
Jeu de test         : 92,253 lignes × 212 colonnes

Taux de défaut train : 8.07%
Taux de défaut test  : 8.07%


## 7. Standardisation

Variables numériques à échelles différentes. StandardScaler sur les variables numériques uniquement. 
Nécessaire pour la régression logistique (sensible aux échelles). 
Sans effet sur Random Forest et XGBoost (insensibles aux échelles), 
mais on standardise une seule fois pour les trois modèles par cohérence.

In [15]:
scaler = StandardScaler()

# Identifier les colonnes numériques (exclure les colonnes binaires issues du one-hot)
colonnes_a_standardiser = X_train.select_dtypes(include='number').columns
colonnes_binaires = [col for col in colonnes_a_standardiser if X_train[col].nunique() <= 2]
colonnes_a_standardiser = [col for col in colonnes_a_standardiser if col not in colonnes_binaires]

print(f"Colonnes à standardiser : {len(colonnes_a_standardiser)}")
print(f"Colonnes binaires (non standardisées) : {len(colonnes_binaires)}")

# Fit sur le train, transform sur train et test
X_train[colonnes_a_standardiser] = scaler.fit_transform(X_train[colonnes_a_standardiser])
X_test[colonnes_a_standardiser] = scaler.transform(X_test[colonnes_a_standardiser])

print(f"\nVérification — moyennes du train (doivent être ~0) :")
print(X_train[colonnes_a_standardiser].mean().head(5).round(4))

Colonnes à standardiser : 60
Colonnes binaires (non standardisées) : 32

Vérification — moyennes du train (doivent être ~0) :
CNT_CHILDREN       -0.0
AMT_INCOME_TOTAL   -0.0
AMT_CREDIT          0.0
AMT_ANNUITY         0.0
AMT_GOODS_PRICE     0.0
dtype: float64


## 8. Sauvegarde et synthèse

Sauvegarde des jeux train/test prêts pour la modélisation et récapitulatif de toutes les transformations effectuées.

In [20]:
# Sauvegarde
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

### Récapitulatif des transformations

- **Dataset initial** : 307 511 lignes × 122 colonnes
- **Colonnes supprimées** : 17 (>60 % manquants, caractéristiques immobilières)
- **Lignes supprimées** : 4 (CODE_GENDER = XNA)
- **Valeurs aberrantes** : DAYS_EMPLOYED 365243 remplacé par NaN
- **Imputation** : médiane (numériques), mode (catégorielles)
- **Features dérivées** : AGE, CREDIT_INCOME_RATIO, ANNUITY_INCOME_RATIO, EMPLOYMENT_YEARS
- **Encodage** : one-hot encoding (drop_first=True), 16 catégorielles → 105 colonnes binaires
- **Standardisation** : StandardScaler sur 60 colonnes numériques
- **Découpage** : 70/30 stratifié (random_state=42)
- **Jeu d'entraînement final** : 215 254 lignes × 212 colonnes
- **Jeu de test final** : 92 253 lignes × 212 colonnes
- **Taux de défaut** : 8,07 % (train et test)